In [0]:
df = spark.table("metadata_governance.silver.silver_metadata_columns")
df.display()

In [0]:
from pyspark.sql.functions import col, when, expr

tier2_fields = ["column_desc", "table_desc", "data_steward", 
                 "security_classification", "term_subdomain", "certification_level"]

completeness_expr = " + ".join([f"CASE WHEN {f} IS NOT NULL THEN 1 ELSE 0 END" for f in tier2_fields])

column_detail = df.withColumn(
    "tier2_filled_count", expr(completeness_expr)
).withColumn(
    "row_completeness_pct", (col("tier2_filled_count") / len(tier2_fields)) * 100
).withColumn(
    "pii_non_compliant",
    when((col("pii_flag") == True) & (col("security_classification").isNull()), True).otherwise(False)
).withColumn(
    "unowned",
    when(col("data_steward").isNull(), True).otherwise(False)
).withColumn(
    "uncertified",
    when(col("certification_level").isNull(), True).otherwise(False)
).select(
    "column_id", "column_name", "table_id", "table_name", "schema_name",
    "database_name", "system_name", "pii_flag", "critical_data_element_flag",
    "row_completeness_pct", "pii_non_compliant", "unowned", "uncertified"
)

column_detail.display()

In [0]:
column_detail.write.mode("overwrite").saveAsTable("metadata_governance.gold.column_governance_detail")

In [0]:
from pyspark.sql.functions import avg, sum as spark_sum, count, round as spark_round

table_summary = column_detail.groupBy("table_id", "table_name", "schema_name", "database_name", "system_name").agg(
    spark_round(avg("row_completeness_pct"), 2).alias("table_completeness_pct"),
    count("column_id").alias("total_columns"),
    spark_sum(col("pii_non_compliant").cast("int")).alias("pii_non_compliant_count"),
    spark_sum(col("unowned").cast("int")).alias("unowned_count"),
    spark_sum(col("uncertified").cast("int")).alias("uncertified_count")
).withColumn(
    "maturity_tier",
    when(col("table_completeness_pct") >= 90, "High")
    .when(col("table_completeness_pct") >= 50, "Medium")
    .otherwise("Low")
)

table_summary.display()

In [0]:
table_summary.write.mode("overwrite").saveAsTable("metadata_governance.gold.table_governance_summary")

In [0]:
spark.sql("SELECT * FROM metadata_governance.gold.table_governance_summary LIMIT 10").display()

In [0]:
spark.sql("SELECT * FROM metadata_governance.gold.column_governance_detail LIMIT 10").display()

In [0]:
%sql
CREATE OR REPLACE TABLE metadata_governance.gold.table_governance_summary_logical AS
SELECT
  COALESCE(system_name,   'unknown_system')   AS system_name,
  COALESCE(database_name, 'unknown_database') AS database_name,
  COALESCE(schema_name,   'unknown_schema')   AS schema_name,
  table_name,
  CONCAT_WS('.',
    COALESCE(system_name,   'unknown_system'),
    COALESCE(database_name, 'unknown_database'),
    COALESCE(schema_name,   'unknown_schema'),
    table_name
  )                                            AS logical_table_key,
  COUNT(*)                                     AS total_columns,
  ROUND(AVG(row_completeness_pct), 2)          AS table_completeness_pct,
  SUM(CASE WHEN pii_non_compliant THEN 1 ELSE 0 END) AS pii_non_compliant_count,
  SUM(CASE WHEN unowned          THEN 1 ELSE 0 END) AS unowned_count,
  SUM(CASE WHEN uncertified      THEN 1 ELSE 0 END) AS uncertified_count,
  CASE
    WHEN AVG(row_completeness_pct) >= 90 THEN 'High'
    WHEN AVG(row_completeness_pct) >= 50 THEN 'Medium'
    ELSE 'Low'
  END                                          AS maturity_tier
FROM metadata_governance.gold.column_governance_detail
GROUP BY
  COALESCE(system_name,   'unknown_system'),
  COALESCE(database_name, 'unknown_database'),
  COALESCE(schema_name,   'unknown_schema'),
  table_name;

In [0]:
%sql
-- 3a. Expect 1,887 (matches the pandas verification)
SELECT COUNT(*) AS logical_tables
FROM metadata_governance.gold.table_governance_summary_logical;

In [0]:
%sql
-- 3b. Must be exactly 10,000 — no columns lost or double-counted
SELECT SUM(total_columns) AS total_columns
FROM metadata_governance.gold.table_governance_summary_logical;

In [0]:
%sql
-- 3c. The 457 nulls should be visible as the unknown bucket
SELECT COUNT(*) AS tables_in_unknown_db, SUM(total_columns) AS cols_in_unknown_db
FROM metadata_governance.gold.table_governance_summary_logical
WHERE database_name = 'unknown_database';


In [0]:
%sql
-- 3d. Columns-per-table shape: expect min 1, max ~15, avg ~5.3
SELECT MIN(total_columns), MAX(total_columns), ROUND(AVG(total_columns),1)
FROM metadata_governance.gold.table_governance_summary_logical;

In [0]:
%sql
-- 3e. The comparison numbers for the demo — new maturity distribution
SELECT maturity_tier, COUNT(*) AS table_count,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM metadata_governance.gold.table_governance_summary_logical
GROUP BY maturity_tier;

In [0]:
%sql
-- 3f. New average completeness (will differ from 75.0 — expected)
SELECT ROUND(AVG(table_completeness_pct), 1)
FROM metadata_governance.gold.table_governance_summary_logical;